# 04 · Wide-and-deep model (Jadon)

**Question this model answers: does adding memorised rules help the neural network?**

Wide-and-deep has two parts, trained together with one loss:

| Part | What it is | Input | What it is good at |
|---|---|---|---|
| **Wide** | A linear layer: one weight per rule, like logistic regression | 0/1 flags + crossed features chosen in `02_eda_interactions_jadon.ipynb` (e.g. tier x recency bucket) | **Memorising** simple rules such as "Gold and ordered in the last 30 days → likely to buy" |
| **Deep** | An MLP | The same inputs as the plain MLP: all 69 features, scaled numbers + one-hot text | **Generalising**: smooth, complex patterns |

`probability = sigmoid( wide(rules) + deep(features) )`

**How the question is answered: an ablation.** Same split, same seed, same training code, with
one part switched off at a time:

| Model | Wide | Deep | Equivalent to |
|---|---|---|---|
| Deep only | off | on | a plain MLP |
| Wide only | on | off | logistic regression on the rules |
| Wide + deep | on | on | my model |

If wide + deep beats deep only by more than the seed-to-seed noise, the memorised rules help.

| Section | What it does |
|---|---|
| 0 | Setup, data, shared split |
| 1 | Build the two inputs (fitted on train only) |
| 2 | First run of the three ablation models |
| 3 | Tuning: crosses, deep-part size, learning rate (every run logged) |
| 4 | Stability: the ablation repeated over 3 seeds |
| 5 | Calibration: make 0.7 mean "about 70% buy" |
| 6 | Final evaluation on test (used once), against the shared baselines |
| 7 | What the wide part memorised |
| 8 | Save, and findings |

## 0. Setup

In [ ]:
import json
import sys
import time
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src" / "data_prep.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import average_precision_score, brier_score_loss, precision_recall_curve

# Shared team code: same split, same preprocessing, same metrics as every other model
from data_prep import (MODELS_DIR, RESULTS_DIR, SEED, best_f1_threshold, build_preprocessor,
                       evaluate, get_splits, load_data, precision_at_top, split_xy)
# My model
from wide_deep import (PlattCalibrator, TrainConfig, WideFeatures, predict_logit, save_model,
                       train_model, wide_rules)

torch.set_num_threads(4)
FIGURES = ROOT / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 50)

In [ ]:
df = load_data()
train, val, test = get_splits(df)          # the one shared 70/15/15 split, seed 42
X_train, y_train = split_xy(train)
X_val, y_val = split_xy(val)
X_test, y_test = split_xy(test)
print(f"train {len(y_train):,} | val {len(y_val):,} | test {len(y_test):,} | buy rate {y_train.mean():.3f}")

## 1. The two inputs

- **Deep input:** `build_preprocessor()` from `src/data_prep.py`, the same one the baselines and
  the plain MLP use (median imputation, scaling, one-hot). Fitted on **train only**.
- **Wide input:** `WideFeatures` from `src/wide_deep.py`, built from the crosses the EDA kept.
  Bucket edges and the list of combinations are learned on **train only**. A combination with
  fewer than 30 train customers is merged into "infrequent", so no rule is learned from a handful
  of people.

In [ ]:
decision = json.loads((RESULTS_DIR / "wide_deep_crosses_jadon.json").read_text())
EDA_CROSSES = [tuple(pair) for pair in decision["crosses"]]
print("Crosses kept by the EDA (best first):")
for a, b in EDA_CROSSES:
    print(f"  {a} x {b}")

# Deep input: identical to the other members' models
deep_pre = build_preprocessor(X_train).fit(X_train)
D_train, D_val, D_test = (np.asarray(deep_pre.transform(X), dtype=np.float32)
                          for X in (X_train, X_val, X_test))

def wide_inputs(crosses):
    # Fit the wide features on TRAIN, then transform all three parts with what train taught it.
    wf = WideFeatures(crosses).fit(X_train)
    return wf, wf.transform(X_train), wf.transform(X_val), wf.transform(X_test)

wide_all = wide_inputs(EDA_CROSSES)
print(f"\nDeep input: {D_train.shape[1]} columns | Wide input with all EDA crosses: {wide_all[1].shape[1]} rules")

## 2. First run of the three ablation models

Starting settings (not tuned yet): deep part 64 → 32, dropout 0.2, Adam learning rate 1e-3,
batch 512, early stopping on **validation PR-AUC** with patience 8. Every run, here and in
Section 3, is added to the tuning log.

In [ ]:
tuning_log = []          # one row per training run: the evidence of optimisation

def run(name, crosses, config, wide=None, seed=SEED, log=True):
    # Train one model and record its settings and validation PR-AUC.
    wf, W_tr, W_va, W_te = wide if wide is not None else wide_inputs(crosses)
    start = time.perf_counter()
    model, history = train_model((W_tr, D_train), y_train, (W_va, D_val), y_val, config, seed=seed)
    row = {"run": len(tuning_log) + 1, "model": name, "seed": seed,
           "crosses": len(crosses) if config.use_wide else 0,
           "wide_rules": W_tr.shape[1] if config.use_wide else 0,
           **config.as_dict(), "best_epoch": model.best_epoch, "epochs_run": len(history),
           "val_pr_auc": history["val_pr_auc"].max(), "seconds": round(time.perf_counter() - start, 1)}
    if log:
        tuning_log.append(row)
    print(f"{name:<40} val PR-AUC {row['val_pr_auc']:.4f}  (best epoch {model.best_epoch}, {row['seconds']}s)")
    return {"model": model, "history": history, "wide": (wf, W_tr, W_va, W_te), "row": row}

base = TrainConfig()
first = {
    "Deep only": run("Deep only", EDA_CROSSES, TrainConfig(use_wide=False), wide=wide_all),
    "Wide only": run("Wide only", EDA_CROSSES, TrainConfig(use_deep=False), wide=wide_all),
    "Wide + deep": run("Wide + deep", EDA_CROSSES, base, wide=wide_all),
}

In [ ]:
# Learning curves for the wide + deep run: is it over-fitting?
h = first["Wide + deep"]["history"]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(h["epoch"], h["train_loss"], label="train loss (weighted)", color="#2a78d6")
ax1.plot(h["epoch"], h["val_loss"], label="validation loss", color="#eb6834")
ax1.set_xlabel("epoch"); ax1.set_title("Loss", loc="left", fontweight="bold"); ax1.legend(frameon=False)
ax2.plot(h["epoch"], h["val_pr_auc"], color="#1baf7a")
ax2.axvline(first["Wide + deep"]["model"].best_epoch, color="#52514e", linestyle="--", label="best epoch (kept)")
ax2.set_xlabel("epoch"); ax2.set_title("Validation PR-AUC", loc="left", fontweight="bold"); ax2.legend(frameon=False)
for ax in (ax1, ax2):
    ax.spines[["top", "right"]].set_visible(False); ax.grid(color="#e4e3df")
fig.tight_layout(); fig.savefig(FIGURES / "jadon_20_learning_curves.png", dpi=150); plt.show()

**How to read it.** If validation loss starts rising while train loss keeps falling, the model is
memorising the train customers (over-fitting). Early stopping keeps the weights from the dashed
line, the epoch with the best validation PR-AUC, not the last epoch.

## 3. Tuning

Three things are tuned, one at a time, each step keeping the best setting from the step before
(a greedy search: 11 runs instead of the hundreds a full grid would need, and each step is easy
to explain). Every choice is made on **validation PR-AUC**; the test split is not touched.

1. **Which crosses** go in the wide part: none (flags only), the top 1, top 3, or all the EDA kept.
2. **Deep-part size:** 32, 64 → 32, 128 → 64.
3. **Learning rate:** 3e-4, 1e-3, 3e-3.
4. **Weight decay on the wide part:** how hard rules backed by little data are shrunk to zero.

In [ ]:
def best_of(results):
    return max(results, key=lambda k: results[k]["row"]["val_pr_auc"])

# Step 1: which crosses
cross_options = {"flags only": [], "top 1": EDA_CROSSES[:1], "top 3": EDA_CROSSES[:3], "all EDA": EDA_CROSSES}
step1 = {k: run(f"W+D crosses={k}", v, base) for k, v in cross_options.items() if k != "all EDA"}
step1["all EDA"] = first["Wide + deep"]          # already trained in Section 2 with these settings
best_crosses_name = best_of(step1)
best_crosses = cross_options[best_crosses_name]
best_wide = step1[best_crosses_name]["wide"]
print("-> best crosses:", best_crosses_name)

In [ ]:
# Step 2: deep-part size (with the best crosses)
cfg = base
step2 = {(64, 32): step1[best_crosses_name]}
for hidden in [(32,), (128, 64)]:
    step2[hidden] = run(f"W+D hidden={'-'.join(map(str, hidden))}", best_crosses,
                        TrainConfig(hidden=hidden), wide=best_wide)
best_hidden = best_of(step2)
cfg = TrainConfig(hidden=best_hidden)
print("-> best deep part:", best_hidden)

# Step 3: learning rate
step3 = {1e-3: step2[best_hidden]}
for lr in [3e-4, 3e-3]:
    step3[lr] = run(f"W+D lr={lr:g}", best_crosses, TrainConfig(hidden=best_hidden, lr=lr), wide=best_wide)
best_lr = best_of(step3)
print("-> best learning rate:", best_lr)

# Step 4: weight decay on the wide part
step4 = {1e-4: step3[best_lr]}
for wl2 in [1e-5, 1e-3]:
    step4[wl2] = run(f"W+D wide_l2={wl2:g}", best_crosses,
                     TrainConfig(hidden=best_hidden, lr=best_lr, wide_l2=wl2), wide=best_wide)
best_wl2 = best_of(step4)
print("-> best wide weight decay:", best_wl2)

BEST_CFG = TrainConfig(hidden=best_hidden, lr=best_lr, wide_l2=best_wl2)
print("\nChosen settings:", BEST_CFG.as_dict(), "| crosses:", best_crosses_name)

In [ ]:
log = pd.DataFrame(tuning_log)
log.to_csv(RESULTS_DIR / "wide_deep_jadon_tuning_log.csv", index=False)
show = ["run", "model", "crosses", "wide_rules", "hidden", "lr", "wide_l2", "best_epoch", "val_pr_auc", "seconds"]
display(log[show].style.format({"val_pr_auc": "{:.4f}", "lr": "{:g}", "wide_l2": "{:g}"})
        .highlight_max(subset="val_pr_auc", color="#d6e8fa").hide(axis="index"))
print("Saved: results/wide_deep_jadon_tuning_log.csv")

**Reading the log.** Differences in the third decimal place (for example 0.561 vs 0.563) may be
noise from the random starting weights. Section 4 measures how big that noise is, so the
comparison in the findings is not over-claimed.

## 4. Stability: the ablation over 3 seeds

One run per model can be lucky. Each of the three ablation models is trained with the chosen
settings and 3 different seeds. The answer to the question is the **mean difference** between
wide + deep and deep only, compared with the **standard deviation** across seeds.

In [ ]:
SEEDS = [42, 7, 2024]
ablation = {"Deep only": TrainConfig(hidden=best_hidden, lr=best_lr, use_wide=False),
            "Wide only": TrainConfig(hidden=best_hidden, lr=best_lr, wide_l2=best_wl2, use_deep=False),
            "Wide + deep": BEST_CFG}
seed_runs = {name: [run(f"{name} seed={s}", best_crosses, c, wide=best_wide, seed=s, log=False)
                    for s in SEEDS] for name, c in ablation.items()}

stab = pd.DataFrame({name: [r["row"]["val_pr_auc"] for r in runs] for name, runs in seed_runs.items()},
                    index=[f"seed {s}" for s in SEEDS])
display(stab.agg(["mean", "std"]).T.style.format("{:.4f}"))

gain = stab["Wide + deep"] - stab["Deep only"]          # paired: same seed, same batches
print(f"Wide + deep minus deep only: {gain.mean():+.4f} on average "
      f"(per seed: {', '.join(f'{g:+.4f}' for g in gain)})")
noise = stab["Deep only"].std()
verdict = ("larger than" if abs(gain.mean()) > 2 * noise else "within")
print(f"Seed-to-seed noise (std of deep only): {noise:.4f} -> the gain is {verdict} 2x the noise.")

## 5. Calibration

The agent reads the probability directly ("0.7 means about 70% buy"), so it must be calibrated.
**Platt scaling** fits a one-input logistic regression from the model's raw score to the outcome,
on the **validation** split. It only rescales the scores, so the ranking (and PR-AUC, ROC-AUC)
does not change. The final model is the seed-42 wide + deep run with the chosen settings.

In [ ]:
final = seed_runs["Wide + deep"][0]                 # seed 42
final_model = final["model"]
wf, W_tr, W_va, W_te = final["wide"]

val_logit = predict_logit(final_model, W_va, D_val)
calibrator = PlattCalibrator().fit(val_logit, y_val)     # fitted on VALIDATION only
val_prob = calibrator.predict_proba(val_logit)

threshold = best_f1_threshold(y_val, val_prob)           # threshold chosen on VALIDATION
print(f"Decision threshold (best F1 on validation): {threshold:.3f}")

## 6. Final evaluation on the test split (used once)

Everything above is frozen. The test split is scored once, with the threshold from validation,
by the same `evaluate()` every member uses. The baselines come from `results/baseline_results.csv`
(made by `01_baselines.ipynb`), so all models are on identical rows.

In [ ]:
test_scores = {}
my_rows = []
for name, runs in seed_runs.items():
    r = runs[0]                                           # seed 42 of each ablation model
    _, _, Wv, Wt = r["wide"]
    if name == "Wide + deep":
        t_score = calibrator.predict_proba(predict_logit(r["model"], Wt, D_test))
        v_score, thr = val_prob, threshold
    else:
        v_score = predict_logit(r["model"], Wv, D_val)
        t_score = predict_logit(r["model"], Wt, D_test)
        thr = best_f1_threshold(y_val, v_score)
    test_scores[name] = t_score
    label = f"{name} (jadon)"
    my_rows.append({"model": label, "split": "val", **evaluate(y_val, v_score, thr)})
    my_rows.append({"model": label, "split": "test", **evaluate(y_test, t_score, thr)})

mine = pd.DataFrame(my_rows)
mine.to_csv(RESULTS_DIR / "wide_deep_jadon_results.csv", index=False)

baselines = pd.read_csv(RESULTS_DIR / "baseline_results.csv")
cols = ["pr_auc", "roc_auc", "precision", "recall", "f1", "precision_top10", "lift_top10"]
table = pd.concat([baselines, mine])
table = table[table["split"] == "test"].set_index("model")[cols + ["accuracy"]]
print(f"TEST split: {len(y_test):,} customers, base rate {y_test.mean():.3f}. Accuracy only for the 85.4% floor.")
display(table.astype(float).style.format("{:.3f}", na_rep="-").format({"lift_top10": "{:.2f}x"})
        .highlight_max(subset=["pr_auc"], color="#d6e8fa"))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
colours = {"Wide + deep": "#2a78d6", "Deep only": "#eb6834", "Wide only": "#1baf7a"}
for name, colour in colours.items():
    p, r, _ = precision_recall_curve(y_test, test_scores[name])
    ax.plot(r, p, color=colour, linewidth=2,
            label=f"{name} (PR-AUC {average_precision_score(y_test, test_scores[name]):.3f})")
ax.axhline(y_test.mean(), color="#52514e", linestyle="--", linewidth=1, label=f"random ({y_test.mean():.3f})")
ax.set_xlabel("Recall (share of buyers found)"); ax.set_ylabel("Precision (share of flagged who bought)")
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title("Precision-recall on the test split", loc="left", fontweight="bold")
ax.spines[["top", "right"]].set_visible(False); ax.grid(color="#e4e3df"); ax.legend(frameon=False, fontsize=8)
fig.tight_layout(); fig.savefig(FIGURES / "jadon_21_pr_curves.png", dpi=150); plt.show()

In [ ]:
# Reliability diagram on TEST: after calibration, do customers given ~0.7 buy ~70% of the time?
raw_test = 1 / (1 + np.exp(-predict_logit(final_model, W_te, D_test)))
cal_test = test_scores["Wide + deep"]
bins = np.linspace(0, 1, 11)
fig, ax = plt.subplots(figsize=(5.5, 5.5))
for label, p, colour in [("before calibration", raw_test, "#c9c8c3"), ("after (Platt)", cal_test, "#2a78d6")]:
    idx = np.clip(np.digitize(p, bins) - 1, 0, 9)
    pred = [p[idx == b].mean() for b in range(10) if (idx == b).sum() >= 20]
    real = [y_test[idx == b].mean() for b in range(10) if (idx == b).sum() >= 20]
    ax.plot(pred, real, marker="o", color=colour, linewidth=2,
            label=f"{label} (Brier {brier_score_loss(y_test, p):.4f})")
ax.plot([0, 1], [0, 1], color="#52514e", linestyle="--", linewidth=1, label="perfect")
ax.set_xlabel("Predicted probability"); ax.set_ylabel("Share who actually bought")
ax.set_title("Calibration on the test split", loc="left", fontweight="bold")
ax.spines[["top", "right"]].set_visible(False); ax.grid(color="#e4e3df"); ax.legend(frameon=False, fontsize=8)
fig.tight_layout(); fig.savefig(FIGURES / "jadon_22_calibration.png", dpi=150); plt.show()

In [ ]:
# In business terms: buyers among the top 1,000 customers each model would contact.
TOP_N = 1000
print(f"Random 1,000 customers: ~{TOP_N * y_test.mean():.0f} buyers")
for name in ["Wide + deep", "Deep only", "Wide only"]:
    found = precision_at_top(y_test, test_scores[name], TOP_N / len(y_test)) * TOP_N
    print(f"{name:<12}: {found:.0f} of the top {TOP_N:,} bought")

## 7. What the wide part memorised

Each wide input has one weight. A positive weight pushes the customer **towards** buying, on top
of what the deep part says. These are the rules the model memorised, and they are the part of the
model a CRM executive can read.

In [ ]:
rules = wide_rules(final_model, wf.get_feature_names_out())
top_rules = pd.concat([rules.tail(10)[::-1].rename("weight").reset_index(),
                       rules.head(10).rename("weight ").reset_index()], axis=1)
top_rules.columns = ["pushes towards buying", "weight", "pushes away from buying", "weight "]
display(top_rules.style.format({"weight": "{:+.2f}", "weight ": "{:+.2f}"}))

# How much of each customer's score comes from each part (validation split).
wide_part = final_model.wide(torch.as_tensor(W_va)).detach().numpy().ravel()
deep_part = val_logit - wide_part
print(f"Spread (std) of the wide part's contribution: {wide_part.std():.2f}  |  deep part: {deep_part.std():.2f}")

**Reading the names.** `customer_tier x days_since_last_order_Gold | 0-30` means "Gold customer
whose last order was 0-30 days ago". `..._infrequent_sklearn` is the bucket for rare combinations.
The spread line shows which part moves customers' scores more. If the wide part's spread is
small, the deep part already explains most of what the rules would have said.

## 8. Save

In [ ]:
save_model(final_model, MODELS_DIR / "wide_deep_jadon.pt")
joblib.dump({"wide_features": wf, "deep_preprocessor": deep_pre, "calibrator": calibrator,
             "threshold": threshold, "config": BEST_CFG.as_dict(), "crosses": best_crosses},
            MODELS_DIR / "wide_deep_jadon_pipeline.joblib")
for f in ["wide_deep_jadon.pt", "wide_deep_jadon_pipeline.joblib"]:
    print("Saved: models/" + f)

## Findings (write these in your own words after running; you will be asked about them live)

1. **Does adding memorised rules help?** Quote the mean gain of wide + deep over deep only from
   Section 4 and compare it with the seed noise. "It helps by 0.00X, which is smaller/larger than
   the noise" is a complete answer either way.
2. **Against the baselines.** Test PR-AUC versus the recency rule (what CRM does today),
   logistic regression and LightGBM. If LightGBM or logistic regression is level with or ahead of
   wide + deep, say so and explain why (the baseline notebook found the signal is mostly simple
   and additive, which a linear model already captures).
3. **What tuning changed.** Which step in the tuning log moved validation PR-AUC most, and which
   did not matter.
4. **The memorised rules.** Two or three rules from Section 7 in business language, and whether
   they match the EDA.
5. **Calibration.** Brier score before vs after, and whether the reliability curve sits on the
   diagonal. The agent depends on this.
6. **Limits.** One split, three seeds; crosses were chosen on validation, so validation scores are
   a little optimistic and the test numbers are the fair ones.